<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code_12_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==============================================================================
# 0. 필요한 패키지 설치
# ==============================================================================
!pip install pyreadstat

# ==============================================================================
# 1. 구글 드라이브 마운트 및 데이터 로드
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import tempfile
import subprocess
import pyreadstat

from IPython.display import display

# 실제 원자료가 위치한 폴더
DATA_DIR = '/content/drive/MyDrive/MyDrive hdac_data '

# 폴더 및 원자료 확인
if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(
        f'데이터 폴더를 찾을 수 없습니다: {DATA_DIR}'
    )

print('원자료 파일 확인:')
print([repr(f) for f in os.listdir(DATA_DIR) if 'hn24' in f.lower()])

# 원자료 파일 경로
path = os.path.join(DATA_DIR, 'hn24_all.sas7bdat')

if not os.path.exists(path):
    raise FileNotFoundError(
        f'hn24_all.sas7bdat 파일을 찾을 수 없습니다: {path}'
    )

# 분석에 필요한 변수
# DE1_dg는 S2 민감도분석을 위해 추가
cols = [
    'age', 'sex', 'HE_DM_HbA1c', 'HE_HP', 'HE_obe',
    'incm', 'edu', 'BS3_1', 'BD1_11',
    'wt_itvex', 'kstrata', 'psu', 'DE1_dg'
]

df, meta = pyreadstat.read_sas7bdat(
    path,
    usecols=cols
)

df.columns = df.columns.str.lower()

print('\n원자료 불러오기 완료')
print('파일 위치:', path)
print('원자료 크기:', df.shape)


# ==============================================================================
# 2. 변수 정의 및 전처리
# ==============================================================================

# 당뇨병: 0=비유병, 1=유병
df['dm'] = np.where(
    df['he_dm_hba1c'].isin([1, 2]), 0,
    np.where(df['he_dm_hba1c'] == 3, 1, np.nan)
)

# 고혈압: 0=비유병, 1=유병
df['hp'] = np.where(
    df['he_hp'].isin([1, 2, 3]), 0,
    np.where(df['he_hp'] == 4, 1, np.nan)
)

# 성별: 0=여자, 1=남자
df['sex_cat'] = np.where(
    df['sex'] == 2, 0,
    np.where(df['sex'] == 1, 1, np.nan)
)

# 연령: 만 19세 이상
df['age_num'] = np.where(
    df['age'] >= 19, df['age'], np.nan
)

# 비만: 0=비만 아님, 1=비만
df['obe_cat'] = np.where(
    df['he_obe'].isin([1, 2, 3]), 0,
    np.where(df['he_obe'].isin([4, 5, 6]), 1, np.nan)
)

# 소득수준
df['incm_cat'] = df['incm'].where(
    df['incm'].isin([1, 2, 3, 4]), np.nan
)

# 교육수준
df['edu_cat'] = df['edu'].where(
    df['edu'].isin([1, 2, 3, 4]), np.nan
)

# 흡연: 0=비흡연, 1=현재 흡연
df['smk_cat'] = np.where(
    df['bs3_1'].isin([3, 8]), 0,
    np.where(df['bs3_1'].isin([1, 2]), 1, np.nan)
)

# 음주: 0=월 1회 미만/안마심, 1=월 1회 이상
df['drk_cat'] = np.where(
    df['bd1_11'].isin([1, 2, 8]), 0,
    np.where(df['bd1_11'].isin([3, 4, 5, 6]), 1, np.nan)
)


# ==============================================================================
# 3. 분석 대상 및 완전 사례 정의
# ==============================================================================

# 분석 대상: 만 19세 이상, 당뇨병 및 고혈압 변수 확인
df['in_analysis'] = (
    (df['age'] >= 19) &
    df['dm'].notna() &
    df['hp'].notna()
)

# 모형 2 보정변수 완전 사례
covars = [
    'age_num', 'sex_cat', 'incm_cat',
    'edu_cat', 'smk_cat', 'drk_cat', 'obe_cat'
]

df['complete_case'] = (
    df['in_analysis'] &
    df[covars].notna().all(axis=1)
)

print('\n[분석 대상 확인]')
print('Table 1 분석 대상:', int(df['in_analysis'].sum()))
print('모형 2 완전 사례:', int(df['complete_case'].sum()))

print('\n[DE1_dg 응답값 확인]')
display(
    df['de1_dg'].value_counts(dropna=False).sort_index()
)


# ==============================================================================
# 4. R 복합표본 분석 환경 준비
# ==============================================================================

!apt-get -qq install -y r-base

!Rscript -e "if (!requireNamespace('survey', quietly=TRUE)) install.packages('survey', repos='https://cloud.r-project.org')"

temp_dir = tempfile.mkdtemp()

input_csv = os.path.join(temp_dir, 'analysis_data.csv')
output_csv = os.path.join(temp_dir, 'table3_result.csv')
r_script = os.path.join(temp_dir, 'table3_analysis.R')

# Python 자료를 R로 전달
df.to_csv(input_csv, index=False)


# ==============================================================================
# 5. R 복합표본 설계 및 민감도분석
# ==============================================================================

r_code = r'''
library(survey)

options(survey.lonely.psu = "adjust")

args <- commandArgs(trailingOnly=TRUE)

dat <- read.csv(
  args[1],
  stringsAsFactors=FALSE,
  na.strings=c("", "NA")
)

# --------------------------------------------------
# 5-1. 복합표본 설계변수 확인
# --------------------------------------------------

design_vars <- c("wt_itvex", "kstrata", "psu")

if (!all(design_vars %in% names(dat))) {
  stop("복합표본 설계변수가 누락되었습니다.")
}

dat <- dat[
  complete.cases(dat[, design_vars]) &
  is.finite(dat$wt_itvex) &
  dat$wt_itvex > 0,
]

dat$dm_f <- factor(
  dat$dm,
  levels=c(0, 1)
)

# --------------------------------------------------
# 5-2. 복합표본 설계 설정
# --------------------------------------------------

design_all <- svydesign(
  ids=~psu,
  strata=~kstrata,
  weights=~wt_itvex,
  data=dat,
  nest=TRUE
)

# 만 19세 이상 분석 대상
design_t1 <- subset(
  design_all,
  age >= 19 &
  !is.na(dm) &
  !is.na(hp) &
  dm %in% c(0, 1) &
  hp %in% c(0, 1)
)

# 모형 2 완전 사례
design_m2 <- subset(
  design_t1,
  !is.na(age_num) &
  !is.na(sex_cat) &
  !is.na(incm_cat) &
  !is.na(edu_cat) &
  !is.na(smk_cat) &
  !is.na(drk_cat) &
  !is.na(obe_cat)
)

# --------------------------------------------------
# 5-3. 주 분석: 모형 2
# --------------------------------------------------

model_main <- svyglm(
  hp ~ dm_f +
    age_num +
    factor(sex_cat) +
    factor(incm_cat) +
    factor(edu_cat) +
    factor(smk_cat) +
    factor(drk_cat) +
    factor(obe_cat),
  design=design_m2,
  family=quasibinomial()
)

# --------------------------------------------------
# 5-4. S1: 비만 변수 제외
# --------------------------------------------------

# 주 분석과 동일한 분석 대상자를 유지
model_s1 <- svyglm(
  hp ~ dm_f +
    age_num +
    factor(sex_cat) +
    factor(incm_cat) +
    factor(edu_cat) +
    factor(smk_cat) +
    factor(drk_cat),
  design=design_m2,
  family=quasibinomial()
)

# --------------------------------------------------
# 5-5. S-PR: 포아송 회귀분석
# --------------------------------------------------

model_pr <- svyglm(
  hp ~ dm_f +
    age_num +
    factor(sex_cat) +
    factor(incm_cat) +
    factor(edu_cat) +
    factor(smk_cat) +
    factor(drk_cat) +
    factor(obe_cat),
  design=design_m2,
  family=quasipoisson(link="log")
)

# --------------------------------------------------
# 5-6. 결과 추출 함수
# --------------------------------------------------

get_result <- function(
  model, design, analysis_name, measure,
  term="dm_f1"
) {

  coef_table <- coef(summary(model))

  if (!(term %in% names(coef(model)))) {
    stop(paste("계수를 찾을 수 없습니다:", term))
  }

  beta <- coef(model)[term]

  ci <- confint(
    model,
    parm=term,
    level=0.95
  )

  p_col <- grep(
    "^Pr\\(",
    colnames(coef_table),
    value=TRUE
  )[1]

  p_value <- coef_table[term, p_col]

  data.frame(
    analysis=analysis_name,
    measure=measure,
    estimate=exp(beta),
    lower=exp(ci[1, 1]),
    upper=exp(ci[1, 2]),
    p=p_value,
    N=nrow(design$variables)
  )
}

# --------------------------------------------------
# 5-7. 주 분석 및 S1, S-PR 결과
# --------------------------------------------------

result_main <- get_result(
  model_main,
  design_m2,
  "주 분석: 모형 2",
  "OR"
)

result_s1 <- get_result(
  model_s1,
  design_m2,
  "S1: 비만 변수 제외",
  "OR"
)

result_pr <- get_result(
  model_pr,
  design_m2,
  "S-PR: 포아송 회귀분석",
  "PR"
)

# --------------------------------------------------
# 5-8. S2: 의사 진단 당뇨병 기준 (DE1_dg 반영)
# --------------------------------------------------
# 0: 당뇨병 비유병, 1: 당뇨병 유병
# 8: 비해당(결측치), 9: 모름/무응답(결측치)

DE1_YES <- 1
DE1_NO <- 0

# 의사 진단 여부를 기준으로 당뇨병 변수 생성
dat$dm_dx <- ifelse(
  dat$de1_dg == DE1_YES, 1,
  ifelse(dat$de1_dg == DE1_NO, 0, NA)
)

dat$dm_dx_f <- factor(
  dat$dm_dx,
  levels=c(0, 1)
)

# 새 변수가 포함되도록 복합표본 설계 재생성
design_all_s2 <- svydesign(
  ids=~psu,
  strata=~kstrata,
  weights=~wt_itvex,
  data=dat,
  nest=TRUE
)

design_s2_base <- subset(
  design_all_s2,
  age >= 19 &
  !is.na(hp) &
  hp %in% c(0, 1) &
  !is.na(dm_dx) &
  dm_dx %in% c(0, 1)
)

design_s2 <- subset(
  design_s2_base,
  !is.na(age_num) &
  !is.na(sex_cat) &
  !is.na(incm_cat) &
  !is.na(edu_cat) &
  !is.na(smk_cat) &
  !is.na(drk_cat) &
  !is.na(obe_cat)
)

model_s2 <- svyglm(
  hp ~ dm_dx_f +
    age_num +
    factor(sex_cat) +
    factor(incm_cat) +
    factor(edu_cat) +
    factor(smk_cat) +
    factor(drk_cat) +
    factor(obe_cat),
  design=design_s2,
  family=quasibinomial()
)

result_s2 <- get_result(
  model_s2,
  design_s2,
  "S2: 의사 진단 당뇨병 기준",
  "OR",
  term="dm_dx_f1"
)

# --------------------------------------------------
# 5-9. Table 3 통합
# --------------------------------------------------

final_result <- rbind(
  result_main,
  result_s1,
  result_pr,
  result_s2
)

# 주 분석과의 추정치 차이
main_estimate <- result_main$estimate

final_result$difference_from_main <-
  final_result$estimate - main_estimate

write.csv(
  final_result,
  args[2],
  row.names=FALSE
)

cat("Table 3 분석 완료\n")
cat("주 분석 N:", nrow(design_m2$variables), "\n")
cat("S1 N:", nrow(design_m2$variables), "\n")
cat("S-PR N:", nrow(design_m2$variables), "\n")
cat("S2 N:", nrow(design_s2$variables), "\n")
'''

with open(r_script, 'w', encoding='utf-8') as f:
    f.write(r_code)

# R 분석 실행
run_result = subprocess.run(
    ['Rscript', r_script, input_csv, output_csv],
    capture_output=True,
    text=True
)

print('\n[R 분석 결과]')
print(run_result.stdout)

if run_result.returncode != 0:
    print(run_result.stderr)
    raise RuntimeError(
        '분석 중 오류가 발생했습니다. DE1_dg 코드를 확인하세요.'
    )


# ==============================================================================
# 6. Table 3 결과 출력
# ==============================================================================

table3_result = pd.read_csv(output_csv)

def format_p(p):
    if p < 0.001:
        return '<0.001'
    return f'{p:.3f}'

table3 = pd.DataFrame({
    '분석': table3_result['analysis'],
    '분석 인원(N)': table3_result['N'].astype(int),
    '효과 지표': table3_result['measure'],

    '추정치 (95% CI)': [
        f"{row.estimate:.3f} "
        f"({row.lower:.3f} - {row.upper:.3f})"
        for row in table3_result.itertuples()
    ],

    'p값': [
        format_p(p)
        for p in table3_result['p']
    ],

    '주 분석과의 추정치 차이': [
        f'{x:.3f}'
        for x in table3_result['difference_from_main']
    ]
})

print('\n' + '=' * 90)
print('[D-2. Table 3. 민감도분석 결과]')
print('=' * 90)

display(table3)


# ==============================================================================
# 7. 분석 방법 및 표본 수 확인
# ==============================================================================

print('\n[분석 방법]')
print('자료: 국민건강영양조사 2024년 원시자료')
print('가중치: wt_itvex')
print('층화변수: kstrata')
print('집락변수: psu')
print('분석 방법: R survey 패키지 복합표본 회귀분석')

print('\n[분석 인원]')
display(
    table3[['분석', '분석 인원(N)']]
)

print('\n최종 Table 3 분석 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
원자료 파일 확인:
["'HN24_ALL.sav'", "'hn24_all.sas7bdat'"]

원자료 불러오기 완료
파일 위치: /content/drive/MyDrive/MyDrive hdac_data /hn24_all.sas7bdat
원자료 크기: (6997, 13)

[분석 대상 확인]
Table 1 분석 대상: 5567
모형 2 완전 사례: 5367

[DE1_dg 응답값 확인]


,count
de1_dg,
0.0,5209
1.0,821
8.0,957
9.0,1
NaN,9



[R 분석 결과]
Table 3 분석 완료
주 분석 N: 5367 
S1 N: 5367 
S-PR N: 5367 
S2 N: 5641 


[D-2. Table 3. 민감도분석 결과]


,분석,분석 인원(N),효과 지표,추정치 (95% CI),p값,주 분석과의 추정치 차이
0,주 분석: 모형 2,5367,OR,2.356 (1.919 - 2.892),<0.001,0.000
1,S1: 비만 변수 제외,5367,OR,2.771 (2.268 - 3.386),<0.001,0.415
2,S-PR: 포아송 회귀분석,5367,PR,1.433 (1.322 - 1.554),<0.001,-0.923
3,S2: 의사 진단 당뇨병 기준,5641,OR,2.193 (1.731 - 2.779),<0.001,-0.163



[분석 방법]
자료: 국민건강영양조사 2024년 원시자료
가중치: wt_itvex
층화변수: kstrata
집락변수: psu
분석 방법: R survey 패키지 복합표본 회귀분석

[분석 인원]


,분석,분석 인원(N)
0,주 분석: 모형 2,5367
1,S1: 비만 변수 제외,5367
2,S-PR: 포아송 회귀분석,5367
3,S2: 의사 진단 당뇨병 기준,5641



최종 Table 3 분석 완료
